# Customer Demographics Study using Data Analytics and Machine Learning

This notebook studies customer demographic patterns and builds machine learning models for demographic-based customer understanding. The workflow includes data cleaning, preprocessing, exploratory data analysis, feature engineering, predictive modeling, customer segmentation, dashboard visualizations, and final insights.

The code is designed to automatically adapt to the demographic columns available in the dataset, such as age, gender, marital status, education, occupation, income, location, family size, spending score, customer segment, purchase frequency, and lifestyle indicators.

## 1. Import Libraries

We use beginner-friendly and commonly available Python libraries for data analysis, visualization, preprocessing, machine learning, and clustering.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
pd.set_option('display.max_columns', None)

## 2. Load Dataset

The loader searches for CSV or Excel files in the current notebook folder and common Colab Drive folders. If several files are present, it gives priority to filenames that mention customer, demographic, profile, or segment data.

In [ ]:
# Optional: set DATA_PATH if your file is stored outside the notebook folder.
# Example: DATA_PATH = '/content/drive/MyDrive/customer_demographics.csv'
DATA_PATH = None


def find_dataset_file(data_path=None):
    """Find a likely customer demographics dataset file."""
    if data_path and os.path.exists(data_path):
        return data_path

    search_folders = ['.', '/content', '/content/drive/MyDrive']
    allowed_extensions = ('.csv', '.xlsx', '.xls')
    candidates = []

    for folder in search_folders:
        if not os.path.exists(folder):
            continue
        for root, dirs, files in os.walk(folder):
            # Keep the search quick in very large folders.
            if root.count(os.sep) - folder.count(os.sep) > 3:
                continue
            for file in files:
                lower_file = file.lower()
                if lower_file.endswith(allowed_extensions):
                    full_path = os.path.join(root, file)
                    candidates.append(full_path)

    if not candidates:
        raise FileNotFoundError(
            'No CSV or Excel dataset file was found. Place the customer demographics dataset in the notebook folder, '
            'or set DATA_PATH to the exact file path.'
        )

    priority_words = ['customer', 'demographic', 'profile', 'segment', 'income', 'lifestyle']
    candidates = sorted(
        candidates,
        key=lambda path: (not any(word in os.path.basename(path).lower() for word in priority_words), len(path))
    )
    return candidates[0]


def load_dataset(path):
    """Load CSV or Excel data into a pandas DataFrame."""
    if path.lower().endswith('.csv'):
        return pd.read_csv(path)
    return pd.read_excel(path)


dataset_path = find_dataset_file(DATA_PATH)
df = load_dataset(dataset_path)

# Standardize column names early so automatic detection stays consistent.
df.columns = df.columns.str.strip()

print('Dataset loaded from:', dataset_path)
print('Dataset shape:', df.shape)
df.head()

## 3. Dataset Overview

The first inspection shows the number of records, available columns, data types, and a small preview of the data. This helps us understand which demographic fields are available before cleaning.

In [ ]:
print('Rows and columns:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)

df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 4. Column Detection

Because demographic datasets may use slightly different column names, the helper functions below identify the most likely columns for age, gender, income, education, occupation, region, family size, spending score, customer segment, purchase frequency, and lifestyle indicators.

In [ ]:
def normalize_name(name):
    """Convert a column name to a simple searchable format."""
    return ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(name)).strip('_')


normalized_columns = {col: normalize_name(col) for col in df.columns}


def find_column(keywords, exclude_keywords=None):
    """Return the first column whose normalized name contains one of the keywords."""
    exclude_keywords = exclude_keywords or []
    for col, normalized in normalized_columns.items():
        has_keyword = any(keyword in normalized for keyword in keywords)
        has_exclusion = any(keyword in normalized for keyword in exclude_keywords)
        if has_keyword and not has_exclusion:
            return col
    return None


column_map = {
    'age': find_column(['age']),
    'gender': find_column(['gender', 'sex']),
    'marital_status': find_column(['marital', 'marriage']),
    'education': find_column(['education', 'degree', 'qualification']),
    'occupation': find_column(['occupation', 'job', 'profession', 'employment']),
    'income': find_column(['income', 'salary', 'earnings', 'annual_income']),
    'region': find_column(['region', 'location', 'city', 'state', 'area', 'zone']),
    'family_size': find_column(['family_size', 'family', 'household', 'dependents']),
    'spending_score': find_column(['spending_score', 'spend_score', 'score', 'spending'], exclude_keywords=['category']),
    'customer_segment': find_column(['customer_segment', 'segment', 'cluster', 'class'], exclude_keywords=['kmeans']),
    'purchase_frequency': find_column(['purchase_frequency', 'frequency', 'purchases', 'visits']),
    'lifestyle': find_column(['lifestyle', 'interest', 'activity', 'behavior', 'habit'])
}

print('Detected demographic columns:')
for role, col in column_map.items():
    print(f'{role}: {col}')

## 5. Data Cleaning

This section removes duplicates, standardizes text values, converts likely numeric fields, fills missing values, and prepares a clean customer demographics dataset for analysis.

In [ ]:
# Work on a copy so the original loaded data stays unchanged.
df_clean = df.copy()

# Remove exact duplicate rows.
initial_rows = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
removed_duplicates = initial_rows - df_clean.shape[0]
print('Duplicate rows removed:', removed_duplicates)

# Standardize column names by trimming extra spaces.
df_clean.columns = df_clean.columns.str.strip()

# Trim extra spaces from text columns.
text_columns = df_clean.select_dtypes(include='object').columns
for col in text_columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

print('Cleaned shape:', df_clean.shape)

In [ ]:
# Convert detected numeric demographic columns when they are available.
numeric_roles = ['age', 'income', 'family_size', 'spending_score', 'purchase_frequency']
for role in numeric_roles:
    col = column_map.get(role)
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print('Missing values before filling:')
df_clean.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Fill missing values using simple beginner-friendly rules.
# Numeric columns use the median. Categorical columns use the most common value.
numeric_columns = df_clean.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df_clean.select_dtypes(exclude=np.number).columns.tolist()

for col in numeric_columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_columns:
    if df_clean[col].mode().empty:
        df_clean[col] = df_clean[col].fillna('Unknown')
    else:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('Missing values after filling:', int(df_clean.isnull().sum().sum()))

## 6. Outlier Detection

Outliers are checked for numeric demographic variables using the Interquartile Range method. Instead of deleting customer records, extreme numeric values are capped to reduce their effect on model training while keeping the dataset complete.

In [ ]:
outlier_summary = []

for col in numeric_columns:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
    outlier_summary.append({'Column': col, 'Outliers': int(outlier_count), 'Lower Bound': lower_bound, 'Upper Bound': upper_bound})

if outlier_summary:
    outlier_summary_df = pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)
else:
    outlier_summary_df = pd.DataFrame(columns=['Column', 'Outliers', 'Lower Bound', 'Upper Bound'])

outlier_summary_df

In [ ]:
# Cap numeric outliers using IQR bounds.
df_processed = df_clean.copy()

for item in outlier_summary:
    col = item['Column']
    df_processed[col] = df_processed[col].clip(lower=item['Lower Bound'], upper=item['Upper Bound'])

if outlier_summary:
    print('Outlier capping completed for numeric columns.')
else:
    print('No numeric columns were available for outlier capping.')
df_processed.head()

## 7. Feature Engineering

New demographic features are created from available columns. These features make the analysis easier to interpret and give machine learning models more meaningful inputs.

In [ ]:
def make_quantile_category(series, labels):
    """Create balanced categories from a numeric column when possible."""
    unique_count = series.nunique(dropna=True)
    if unique_count < 2:
        return pd.Series([labels[0]] * len(series), index=series.index)
    bins = min(len(labels), unique_count)
    try:
        return pd.qcut(series.rank(method='first'), q=bins, labels=labels[:bins])
    except ValueError:
        return pd.cut(series, bins=bins, labels=labels[:bins], include_lowest=True)


age_col = column_map.get('age')
income_col = column_map.get('income')
spending_col = column_map.get('spending_score')
family_col = column_map.get('family_size')
occupation_col = column_map.get('occupation')
frequency_col = column_map.get('purchase_frequency')

# Age groups show life-stage based demographic patterns.
if age_col in df_processed.columns:
    df_processed['Age_Group'] = pd.cut(
        df_processed[age_col],
        bins=[0, 18, 25, 35, 45, 60, np.inf],
        labels=['Teen', 'Young Adult', 'Adult', 'Mid Age', 'Senior Adult', 'Senior'],
        include_lowest=True
    )

# Income categories describe customer economic level.
if income_col in df_processed.columns:
    df_processed['Income_Category'] = make_quantile_category(
        df_processed[income_col],
        ['Low Income', 'Middle Income', 'High Income', 'Premium Income']
    )

# Spending categories show customer engagement or value potential.
if spending_col in df_processed.columns:
    df_processed['Spending_Category'] = make_quantile_category(
        df_processed[spending_col],
        ['Low Spending', 'Moderate Spending', 'High Spending', 'Very High Spending']
    )

# Family type converts household size into interpretable groups.
if family_col in df_processed.columns:
    df_processed['Family_Type'] = pd.cut(
        df_processed[family_col],
        bins=[-np.inf, 1, 3, 5, np.inf],
        labels=['Single', 'Small Family', 'Medium Family', 'Large Family']
    )

# Purchase frequency categories describe activity level.
if frequency_col in df_processed.columns:
    df_processed['Purchase_Frequency_Group'] = make_quantile_category(
        df_processed[frequency_col],
        ['Rare', 'Occasional', 'Frequent', 'Very Frequent']
    )

# Occupation groups simplify highly detailed job titles.
if occupation_col in df_processed.columns:
    top_occupations = df_processed[occupation_col].value_counts().head(6).index
    df_processed['Occupation_Group'] = np.where(df_processed[occupation_col].isin(top_occupations), df_processed[occupation_col], 'Other')

# Customer value tier combines spending and income when available.
if spending_col in df_processed.columns:
    score_source = df_processed[spending_col]
elif income_col in df_processed.columns:
    score_source = df_processed[income_col]
else:
    numeric_for_value = df_processed.select_dtypes(include=np.number).columns
    score_source = df_processed[numeric_for_value[0]] if len(numeric_for_value) > 0 else pd.Series([0] * len(df_processed))

df_processed['Customer_Value_Tier'] = make_quantile_category(
    score_source,
    ['Low Value', 'Medium Value', 'High Value', 'Premium Value']
)

df_processed['High_Value_Customer'] = df_processed['Customer_Value_Tier'].astype(str).isin(['High Value', 'Premium Value']).astype(int)

print('New engineered features:')
engineered_features = ['Age_Group', 'Income_Category', 'Spending_Category', 'Family_Type', 'Purchase_Frequency_Group', 'Occupation_Group', 'Customer_Value_Tier', 'High_Value_Customer']
print([col for col in engineered_features if col in df_processed.columns])
df_processed.head()

## 8. Label Encoding and One-Hot Encoding Preview

Label encoding is useful for simple numeric representation of categories. One-hot encoding is used later inside machine learning pipelines so models can learn from categorical demographic variables properly.

In [ ]:
# Label encode categorical columns for quick inspection and correlation-style summaries.
df_label_encoded = df_processed.copy()
label_encoders = {}

for col in df_label_encoded.select_dtypes(exclude=np.number).columns:
    encoder = LabelEncoder()
    df_label_encoded[col] = encoder.fit_transform(df_label_encoded[col].astype(str))
    label_encoders[col] = encoder

print('Label encoding completed for categorical columns.')
df_label_encoded.head()

## 9. Exploratory Data Analysis

The EDA section focuses on demographic distributions, profile differences, spending patterns, customer segments, and numeric relationships. Each visualization checks whether the required column is available before plotting.

In [ ]:
def plot_count(column, title, xlabel=None, top_n=12):
    """Plot a count chart for a categorical column if it exists."""
    if column not in df_processed.columns:
        print(f'Skipped: {title} because the column was not found.')
        return
    order = df_processed[column].value_counts().head(top_n).index
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df_processed, x=column, order=order, palette='Set2')
    plt.title(title)
    plt.xlabel(xlabel or column)
    plt.ylabel('Number of Customers')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()


def plot_hist(column, title, xlabel=None):
    """Plot a histogram for a numeric column if it exists."""
    if column not in df_processed.columns:
        print(f'Skipped: {title} because the column was not found.')
        return
    plt.figure(figsize=(10, 5))
    sns.histplot(df_processed[column], kde=True, color='#4C78A8')
    plt.title(title)
    plt.xlabel(xlabel or column)
    plt.ylabel('Number of Customers')
    plt.tight_layout()
    plt.show()


def plot_box_by_category(category_col, numeric_col, title):
    """Plot a boxplot comparing a numeric variable across demographic groups."""
    if category_col not in df_processed.columns or numeric_col not in df_processed.columns:
        print(f'Skipped: {title} because a required column was not found.')
        return
    plt.figure(figsize=(11, 5))
    sns.boxplot(data=df_processed, x=category_col, y=numeric_col, palette='Set3')
    plt.title(title)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

### Age Distribution

In [ ]:
plot_hist(age_col, 'Age Distribution of Customers', 'Age')

The age distribution helps identify whether the customer base is concentrated among young adults, middle-aged customers, senior customers, or a mix of age groups. This is useful for designing segment-specific communication and services.

### Gender Distribution

In [ ]:
plot_count(column_map.get('gender'), 'Gender Distribution of Customers', 'Gender')

Gender distribution shows the composition of the customer base. A balanced or skewed distribution can guide demographic profiling and help compare engagement across customer groups.

### Income Distribution

In [ ]:
plot_hist(income_col, 'Income Distribution of Customers', 'Income')

Income distribution highlights the economic profile of customers. Understanding income spread helps interpret spending behavior, customer value tiers, and affordability-based segmentation.

### Education Analysis

In [ ]:
plot_count(column_map.get('education'), 'Education Level Distribution', 'Education Level')

Education analysis shows which education groups are most represented in the customer base. These differences may influence product interest, communication preference, and lifestyle behavior.

### Occupation Analysis

In [ ]:
plot_count(column_map.get('occupation'), 'Occupation Distribution', 'Occupation', top_n=10)

Occupation analysis identifies the most common professional backgrounds among customers. This helps connect customer profiles with income potential, lifestyle needs, and spending patterns.

### Marital Status Analysis

In [ ]:
plot_count(column_map.get('marital_status'), 'Marital Status Distribution', 'Marital Status')

Marital status can influence household spending needs, family priorities, and customer preferences. Comparing this variable with income and family size can reveal useful demographic differences.

### Regional Analysis

In [ ]:
plot_count(column_map.get('region'), 'Customer Distribution by Region or Location', 'Region / Location', top_n=12)

Regional analysis shows where customers are concentrated. Location-based differences may explain variations in income, lifestyle, purchase frequency, and spending behavior.

### Spending Pattern Analysis

In [ ]:
plot_hist(spending_col, 'Spending Score Distribution', 'Spending Score')

Spending score distribution helps separate low, moderate, and high-engagement customers. This is one of the most important variables for identifying high-value customer profiles.

### Customer Segment Distribution

In [ ]:
segment_col = column_map.get('customer_segment')
if segment_col in df_processed.columns:
    plot_count(segment_col, 'Customer Segment Distribution', 'Customer Segment')
else:
    plot_count('Customer_Value_Tier', 'Customer Value Tier Distribution', 'Customer Value Tier')

Segment distribution shows how customers are grouped into business-relevant profiles. If an existing segment column is not available, the engineered customer value tier is used as an interpretable segmentation alternative.

### Age Group and Income Category

In [ ]:
if 'Age_Group' in df_processed.columns and 'Income_Category' in df_processed.columns:
    age_income_table = pd.crosstab(df_processed['Age_Group'], df_processed['Income_Category'])
    plt.figure(figsize=(10, 5))
    sns.heatmap(age_income_table, annot=True, fmt='d', cmap='YlGnBu')
    plt.title('Age Group vs Income Category')
    plt.xlabel('Income Category')
    plt.ylabel('Age Group')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped: Age Group vs Income Category because required engineered features were not available.')

This heatmap compares age-based life stages with income levels. It can reveal whether higher income groups are concentrated in specific age ranges or distributed evenly across customers.

### Correlation Heatmap

In [ ]:
numeric_corr = df_label_encoded.select_dtypes(include=np.number).corr()

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_corr, cmap='coolwarm', center=0, linewidths=0.4)
plt.title('Correlation Heatmap of Demographic and Engineered Features')
plt.tight_layout()
plt.show()

The correlation heatmap summarizes relationships among numeric and encoded demographic variables. Strong positive or negative relationships can point to important features for customer prediction and segmentation.

## 10. Advanced Demographic Analysis

This section compares demographic groups and studies the relationships among income, spending, age, and occupation.

### Income vs Spending Analysis

In [ ]:
if income_col in df_processed.columns and spending_col in df_processed.columns:
    plt.figure(figsize=(10, 5))
    hue_col = column_map.get('gender') if column_map.get('gender') in df_processed.columns else None
    sns.scatterplot(data=df_processed, x=income_col, y=spending_col, hue=hue_col, alpha=0.7)
    plt.title('Income vs Spending Score')
    plt.xlabel('Income')
    plt.ylabel('Spending Score')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped: Income vs Spending Analysis because income or spending score was not found.')

Income vs spending analysis helps identify whether higher income customers also show stronger spending engagement. Customers with high income and high spending scores are often strong candidates for premium targeting.

### Age vs Spending Analysis

In [ ]:
if age_col in df_processed.columns and spending_col in df_processed.columns:
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=df_processed, x=age_col, y=spending_col, hue='Customer_Value_Tier', alpha=0.7)
    plt.title('Age vs Spending Score by Customer Value Tier')
    plt.xlabel('Age')
    plt.ylabel('Spending Score')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped: Age vs Spending Analysis because age or spending score was not found.')

Age vs spending analysis shows whether spending behavior changes across life stages. This can reveal age groups that are more active, premium-oriented, or price-sensitive.

### Occupation vs Income Analysis

In [ ]:
occupation_for_plot = 'Occupation_Group' if 'Occupation_Group' in df_processed.columns else occupation_col
plot_box_by_category(occupation_for_plot, income_col, 'Income Distribution by Occupation Group')

Occupation vs income analysis compares earning patterns across professional groups. These differences can help explain customer value tiers and support occupation-based demographic segmentation.

### Demographic Group Comparison

In [ ]:
group_col = None
for possible_col in ['Age_Group', column_map.get('gender'), column_map.get('marital_status'), column_map.get('education'), 'Family_Type']:
    if possible_col in df_processed.columns:
        group_col = possible_col
        break

comparison_metrics = [col for col in [income_col, spending_col, frequency_col, family_col] if col in df_processed.columns]

if group_col and comparison_metrics:
    demographic_comparison = df_processed.groupby(group_col)[comparison_metrics].mean().round(2)
    display(demographic_comparison)
    demographic_comparison.plot(kind='bar', figsize=(11, 5))
    plt.title(f'Demographic Group Comparison by {group_col}')
    plt.ylabel('Average Value')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped: Demographic group comparison because suitable grouping or numeric metrics were not available.')

The demographic group comparison summarizes how key numeric measures vary across customer groups. This makes it easier to identify high-income, high-spending, or highly active demographic profiles.

## 11. Machine Learning Target Selection

The notebook automatically selects a demographic prediction target. It prefers an existing customer segment column, then engineered spending category, customer value tier, income category, and finally high-value customer classification.

In [ ]:
target_candidates = [
    column_map.get('customer_segment'),
    'Spending_Category',
    'Customer_Value_Tier',
    'Income_Category',
    'High_Value_Customer'
]

target_col = None
for candidate in target_candidates:
    if candidate in df_processed.columns and df_processed[candidate].nunique(dropna=True) >= 2:
        target_col = candidate
        break

if target_col is None:
    raise ValueError('No suitable classification target was found. The dataset needs at least one usable segment, spending, income, or value column.')

print('Selected prediction target:', target_col)
print(df_processed[target_col].value_counts())

The selected target represents the main customer demographics prediction task. Depending on the available data, the models predict customer segment, spending category, customer value tier, income group, or high-value customer status.

## 12. Model Data Preparation

Features and target are separated, categorical variables are one-hot encoded, numeric variables are scaled, and small classes are handled carefully so the train-test split remains valid.

In [ ]:
# Prepare a modeling dataset.
model_df = df_processed.copy()

# Remove rows with missing target values if any remain.
model_df = model_df.dropna(subset=[target_col])

# Keep only target classes with at least two records.
class_counts = model_df[target_col].value_counts()
valid_classes = class_counts[class_counts >= 2].index
model_df = model_df[model_df[target_col].isin(valid_classes)].copy()

# Encode target labels.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(model_df[target_col].astype(str))

# Drop target and obvious leakage columns when target was engineered from a source variable.
leakage_columns = [target_col]
if target_col == 'Spending_Category' and spending_col in model_df.columns:
    leakage_columns.append(spending_col)
if target_col == 'Income_Category' and income_col in model_df.columns:
    leakage_columns.append(income_col)
if target_col in ['Customer_Value_Tier', 'High_Value_Customer']:
    for col in [spending_col, income_col, 'Customer_Value_Tier', 'High_Value_Customer']:
        if col in model_df.columns and col != target_col:
            leakage_columns.append(col)

X = model_df.drop(columns=list(dict.fromkeys(leakage_columns)), errors='ignore')

# Remove columns with all unique values because they behave like IDs and do not generalize well.
unique_ratio = X.nunique(dropna=False) / len(X)
id_like_columns = unique_ratio[unique_ratio > 0.95].index.tolist()
X = X.drop(columns=id_like_columns, errors='ignore')

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print('Rows used for modeling:', X.shape[0])
print('Features used for modeling:', X.shape[1])
print('Dropped ID-like columns:', id_like_columns)
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

In [ ]:
# Build preprocessing for machine learning.
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop'
)

# Stratification is used only when every class has enough records.
stratify_values = y if pd.Series(y).value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=stratify_values
)

print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)

## 13. Train Classification Models

Four beginner-friendly classification models are trained and compared: Logistic Regression, Decision Tree, Random Forest, and K-Nearest Neighbors.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

trained_models = {}
model_results = []

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    trained_models[model_name] = pipeline
    model_results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    })

model_results_df = pd.DataFrame(model_results).sort_values('F1 Score', ascending=False)
model_results_df

The model comparison table shows which algorithm performs best for the selected demographic prediction target. F1 Score is especially useful because it balances precision and recall across classes.

### Model Performance Visualization

In [ ]:
plt.figure(figsize=(11, 5))
model_results_melted = model_results_df.melt(id_vars='Model', var_name='Metric', value_name='Score')
sns.barplot(data=model_results_melted, x='Model', y='Score', hue='Metric', palette='Set2')
plt.title('Classification Model Performance Comparison')
plt.ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

The performance chart compares accuracy, precision, recall, and F1 score side by side. This helps select the most reliable model for customer segment or value prediction.

### Best Model Evaluation

In [ ]:
best_model_name = model_results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
y_best_pred = best_model.predict(X_test)

print('Best model:', best_model_name)
print('\nClassification Report:')
print(classification_report(y_test, y_best_pred, target_names=target_encoder.classes_, zero_division=0))

cm = confusion_matrix(y_test, y_best_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_encoder.classes_, yticklabels=target_encoder.classes_)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

The confusion matrix shows where the best model predicts correctly and where it confuses one customer group with another. This is valuable for understanding practical model limitations.

### Random Forest Feature Importance

In [ ]:
if 'Random Forest' in trained_models:
    rf_pipeline = trained_models['Random Forest']
    rf_model = rf_pipeline.named_steps['model']

    encoded_feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
    importance_df = pd.DataFrame({
        'Feature': encoded_feature_names,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, y='Feature', x='Importance', palette='viridis')
    plt.title('Top Feature Importance for Customer Prediction')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

    display(importance_df)
else:
    print('Random Forest model was not available.')

Feature importance identifies the demographic variables that contribute most to prediction. These features often become the strongest business signals for customer profiling and targeted strategy.

## 14. Customer Segmentation Using K-Means Clustering

K-Means clustering groups customers based on available demographic and behavioral features. This unsupervised approach can uncover natural customer profiles even when no segment label is provided.

In [ ]:
# Prepare features for clustering.
cluster_features = []
for col in [age_col, income_col, spending_col, family_col, frequency_col]:
    if col in df_processed.columns and col not in cluster_features:
        cluster_features.append(col)

# If only a few demographic numeric columns exist, use all numeric columns as a fallback.
if len(cluster_features) < 2:
    cluster_features = df_processed.select_dtypes(include=np.number).columns.tolist()

cluster_data = df_processed[cluster_features].copy()
cluster_data = cluster_data.fillna(cluster_data.median())

scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_data)

max_k = min(8, len(cluster_data) - 1)
inertia = []
k_values = list(range(2, max_k + 1))

for k in k_values:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(cluster_scaled)
    inertia.append(kmeans_temp.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia, marker='o')
plt.title('Elbow Method for Customer Segmentation')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

print('Features used for clustering:', cluster_features)

The elbow method helps choose a reasonable number of customer clusters. A lower inertia means customers within each cluster are more similar, but too many clusters can make interpretation harder.

In [ ]:
# Use 4 clusters when possible, otherwise adapt to the dataset size.
n_clusters = min(4, max(2, len(cluster_data) // 10)) if len(cluster_data) >= 20 else min(2, len(cluster_data))

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_processed['Customer_Cluster'] = kmeans.fit_predict(cluster_scaled)

cluster_profile = df_processed.groupby('Customer_Cluster')[cluster_features].mean().round(2)
cluster_profile

The cluster profile table describes the average demographic characteristics of each customer group. These clusters can be interpreted as practical customer personas for analysis and strategy.

In [ ]:
if cluster_scaled.shape[1] >= 2 and len(df_processed) >= 3:
    pca = PCA(n_components=2, random_state=42)
    cluster_pca = pca.fit_transform(cluster_scaled)

    plt.figure(figsize=(9, 6))
    sns.scatterplot(x=cluster_pca[:, 0], y=cluster_pca[:, 1], hue=df_processed['Customer_Cluster'], palette='tab10', alpha=0.8)
    plt.title('Customer Segmentation Clusters - PCA View')
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.legend(title='Cluster')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped PCA cluster visualization because there are not enough features or rows.')

The PCA scatter plot projects customer clusters into two dimensions for visualization. Nearby points represent customers with similar demographic and behavioral profiles.

## 15. Dashboard Visualizations

The dashboard brings together the most important demographic views in one place: age, gender, income, spending, value tier, and customer clusters.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# 1. Age distribution
if age_col in df_processed.columns:
    sns.histplot(df_processed[age_col], kde=True, ax=axes[0], color='#4C78A8')
    axes[0].set_title('Age Distribution')
else:
    axes[0].text(0.5, 0.5, 'Age column not available', ha='center', va='center')
    axes[0].set_axis_off()

# 2. Gender distribution
if column_map.get('gender') in df_processed.columns:
    df_processed[column_map.get('gender')].value_counts().plot(kind='bar', ax=axes[1], color='#F58518')
    axes[1].set_title('Gender Distribution')
    axes[1].tick_params(axis='x', rotation=35)
else:
    axes[1].text(0.5, 0.5, 'Gender column not available', ha='center', va='center')
    axes[1].set_axis_off()

# 3. Income distribution
if income_col in df_processed.columns:
    sns.histplot(df_processed[income_col], kde=True, ax=axes[2], color='#54A24B')
    axes[2].set_title('Income Distribution')
else:
    axes[2].text(0.5, 0.5, 'Income column not available', ha='center', va='center')
    axes[2].set_axis_off()

# 4. Spending score distribution
if spending_col in df_processed.columns:
    sns.histplot(df_processed[spending_col], kde=True, ax=axes[3], color='#B279A2')
    axes[3].set_title('Spending Score Distribution')
else:
    axes[3].text(0.5, 0.5, 'Spending score column not available', ha='center', va='center')
    axes[3].set_axis_off()

# 5. Customer value tier
if 'Customer_Value_Tier' in df_processed.columns:
    df_processed['Customer_Value_Tier'].value_counts().plot(kind='bar', ax=axes[4], color='#E45756')
    axes[4].set_title('Customer Value Tier')
    axes[4].tick_params(axis='x', rotation=35)
else:
    axes[4].text(0.5, 0.5, 'Value tier not available', ha='center', va='center')
    axes[4].set_axis_off()

# 6. Customer clusters
if 'Customer_Cluster' in df_processed.columns:
    df_processed['Customer_Cluster'].value_counts().sort_index().plot(kind='bar', ax=axes[5], color='#72B7B2')
    axes[5].set_title('Customer Clusters')
    axes[5].set_xlabel('Cluster')
else:
    axes[5].text(0.5, 0.5, 'Clusters not available', ha='center', va='center')
    axes[5].set_axis_off()

plt.suptitle('Customer Demographics Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

The dashboard provides a compact executive view of the customer base. It summarizes customer composition, economic profile, spending behavior, value tiers, and machine-generated clusters.

## 16. Insights and Conclusion

The final cells automatically summarize the most important findings from the available dataset.

In [ ]:
print('CUSTOMER DEMOGRAPHICS STUDY - KEY INSIGHTS')
print('-' * 55)
print(f'Total customers analyzed: {len(df_processed)}')
print(f'Total columns after feature engineering: {df_processed.shape[1]}')
print(f'Prediction target used: {target_col}')
print(f'Best classification model: {best_model_name}')
print(f'Best model F1 score: {model_results_df.iloc[0]["F1 Score"]:.3f}')

if age_col in df_processed.columns:
    print(f'Average customer age: {df_processed[age_col].mean():.2f}')

if income_col in df_processed.columns:
    print(f'Average income: {df_processed[income_col].mean():.2f}')

if spending_col in df_processed.columns:
    print(f'Average spending score: {df_processed[spending_col].mean():.2f}')

if 'Customer_Value_Tier' in df_processed.columns:
    print('\nCustomer value tier distribution:')
    print(df_processed['Customer_Value_Tier'].value_counts())

if 'Customer_Cluster' in df_processed.columns:
    print('\nCustomer cluster distribution:')
    print(df_processed['Customer_Cluster'].value_counts().sort_index())

### Conclusion

This project transformed the analysis into a complete Customer Demographics Study using Data Analytics and Machine Learning. The notebook cleaned and preprocessed demographic data, engineered meaningful customer profile features, explored demographic patterns, trained multiple classification models, created K-Means customer segments, and produced dashboard-style visualizations.

The final results support customer profiling, segment prediction, high-value customer identification, and demographic group comparison. These outputs can help businesses understand who their customers are, which demographic groups show stronger value potential, and how customer profiles differ across income, age, occupation, region, and spending behavior.